In [1]:
#import torch
#import torch.nn as nn
#import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
from scipy.special import comb

In [71]:
def P_of_Q(Q,L,th):
    # Q: total charge of the initial state
    p = [] # p(0), p(1), p(2) in each cell
    p.append(((np.sin(2*th))**2)/4)
    p.append(1 - ((np.sin(2*th))**2)/2)
    p.append(((np.sin(2*th))**2)/4)
    p = np.array(p)

    Ha, configs = count_and_list_solutions(Q, L//2) # (Q,L/2)
    configs = np.array(configs)

    mask = np.any(configs > 2, axis=1)
    charges = configs[~mask]

    total = np.zeros(charges.shape[0])

    a = charges.shape[0]
    b = charges.shape[1]

    for i in range(a):
        prob = 1
        for j in range(b):
            prob = prob * p[charges[i,j]]
        total[i] = prob

    P = np.sum(total)
    return P

# k: number of replicas
def find_solutions(k, dim_A, current_solution, configs):
    if dim_A == 1:
        current_solution.append(k)
        configs.append(tuple(current_solution))
        current_solution.pop()
    else:
        for i in range(k + 1):
            current_solution.append(i)
            find_solutions(k - i, dim_A - 1, current_solution, configs)
            current_solution.pop()

def count_and_list_solutions(k, dim_A):
    configs = []
    find_solutions(k, dim_A, [], configs)
    return len(configs), configs
# configs is the list of non-negative solutions.

In [72]:
# Get the basis vectors
def get_Z(L):
  dim = 2**L
  basis = np.zeros((dim, L), dtype = int)
  index = np.arange(dim) # this enumerates all states in the Hilbert space
  for site in range(L):
    basis[:,site] = np.mod(index,2) # get 1 bit from the index
    index //= 2 # integer division: move on to the next bit
  basis = basis[:,::-1] # order the sites correctly
  Z = 1-2*basis # basis state 0 has Z=+1, 1 has Z=-1
  return Z

In [73]:
def psi_H(psi,L,C):
  # This function generates a normalized Haar random vector in the charge sector with charge C.
  # L: system size
  # C: charge sector

  Z = (get_Z(L)+1)/2
  Q = Z.sum(axis=1)

  positions = np.where(Q == C)[0]
  mask = np.zeros_like(psi, dtype=bool)
  mask[positions] = True
  new_psi = psi * mask
  new_psi = new_psi / np.linalg.norm(new_psi)

  return new_psi

In [74]:
def construct_probs_array(psi, L, LA):
    """
    Measure the bath and obtain the bitstring z.
    Calculate the probability of each outcome under every possible combination of random Paulis.
    This version only works for LA=2, but can be generalized to larger LA.

    Return：
      P_z       : shape (2**(L-LA),)      — Probability to get z
      basis_list: list of (p1,p2) length 9 — p1,p2 ∈ {2,3,4} correspond to X,Y,Z.
      probs     : shape (2**(L-LA), 9, 2**LA)
    """
    LB = L - LA
    dim_A = 2**LA
    dim_B = 2**LB


    psi_mat = psi.reshape((dim_A, dim_B))
    # P(z) = ||ψ_A(z)||^2
    P_z = np.linalg.norm(psi_mat, axis=0)**2

    # 定義 Pauli 基底（2=X,3=Y,4=Z）的本徵向量矩陣
    pauli_bases = {
        2: np.array([[1,  1],
                     [1, -1]]) / np.sqrt(2),            # X basis
        3: np.array([[1, 1],
                     [1j, -1j]]) / np.sqrt(2),           # Y basis
        4: np.eye(2),                                 # Z basis
    }

    # 9 combinations of random Pauli
    basis_list = [(p1, p2) for p1 in pauli_bases for p2 in pauli_bases]

    probs = np.zeros((dim_B, len(basis_list), dim_A))

    for z in range(dim_B):
        psi_A_z = psi_mat[:, z]
        if P_z[z] == 0:
            continue
        # normalization
        psi_A_norm = psi_A_z / np.sqrt(P_z[z])
        for idx, (p1, p2) in enumerate(basis_list):
            U = np.kron(pauli_bases[p1], pauli_bases[p2])   # 4×4 transformation matrix
            amps = U.conj().T @ psi_A_norm
            probs[z, idx, :] = np.abs(amps)**2              # probabilities of 4 outcomes (00,01,10,11)

    return P_z, basis_list, probs

In [75]:
def sample_dataset(P_z, basis_list, probs, L, LA: int,
                   M_B: int, R_A: int, seed=None):
    """
    Generate a hierarchical dataset with M_B independent outer draws
    from P(z_B) and R_A random-Pauli measurements for each draw.

    Parameters:
    - P_z: 1D array of probabilities for each z in subsystem B
    - basis_list: list of Pauli basis tuples for subsystem A
    - probs: 3D array of conditional outcome probabilities for A
    - LA: number of qubits in subsystem A
    - M_B: number of independent outer draws from P_z; z values may repeat
    - R_A: number of A measurement records for each outer draw
    - seed: seed for a local NumPy random-number generator

    Returns:
    - pandas.DataFrame with columns:
      ['outer_id', 'z0', ..., 'z(LB-1)', 'P1', 'b1', ..., 'P{LA}', 'b{LA}']
    """

    rng = np.random.default_rng(seed)

    LB = L - LA
    dim_A = 2**LA

    # Normalize P_z
    p_norm = P_z / np.sum(P_z)

    # M_B independent outer draws; repeated z values are physically allowed.
    z_list = rng.choice(len(P_z), size=M_B, replace=True, p=p_norm)

    rows = []
    for outer_id, z in enumerate(z_list):
        z_bits = list(map(int, np.binary_repr(z, width=LB)))
        for _ in range(R_A):
            basis_idx = rng.choice(len(basis_list))
            outcome = rng.choice(dim_A, p=probs[z, basis_idx, :])
            outcome_bits = list(map(int, np.binary_repr(outcome, width=LA)))
            p_tuple = basis_list[basis_idx]

            # Interleave P_i and b_i
            interleaved = []
            for p, b in zip(p_tuple, outcome_bits):
                interleaved.extend([p, b])

            rows.append([outer_id] + z_bits + interleaved)

    # Column names
    cols = ["outer_id"] + [f"z{i+1}" for i in range(LB)]
    for i in range(1, LA+1):
        cols += [f"P{i}", f"b{i}"]

    return pd.DataFrame(rows, columns=cols)

In [76]:
L = 12              # Total number of qubits in the system
LA = 2              # Subsystem A (e.g., first 2 qubits)
LB = L - LA         # Bath B (remaining qubits)
dim = 2**L
M_B = LB**3          # Number of independent outer z_B draws
R_A = 4            # Random-Pauli A measurements per outer draw
seed = 48           # Reproduces both the state and the sampled dataset
th = 0              # Neel state

#---------------Create late-time (thermalized) state---------------------
# The initial state is assumed to be the same as that in our deep therm paper.
#psi = np.random.randn(2**L) + 1j * np.random.randn(2**L)
#psi = psi / np.linalg.norm(psi)

state_rng = np.random.default_rng(seed)
thermalized_state = np.zeros(dim, dtype=complex)
for C in range(L+1):
  psi = state_rng.normal(size=2**L) + 1j * state_rng.normal(size=2**L)
  psi = psi / np.linalg.norm(psi)
  psi_H_ = psi_H(psi,L,C)
  psi_H_norm = psi_H_ / np.linalg.norm(psi_H_)
  thermalized_state += np.sqrt(P_of_Q(C,L,th)) * psi_H_norm

# np.save("thermalized_state.npy", thermalized_state)

In [78]:
P_z, basis_list, probs = construct_probs_array(thermalized_state, L, LA)
df_new = sample_dataset(
    P_z, basis_list, probs, L, LA, M_B=M_B, R_A=R_A, seed=seed + 1
)
df_new.to_csv(f"U1_dataset_LA={LA}_LB={LB}_Neel.csv", index=False, header=True)